# 8.1. 대규모 데이터 처리를 위한 Sparse Matrix 사용

만약 Movie Lens 데이터 2M 을 사용한다고 가정해보면.. 

사용자와 영화가 각각 13만개 정도 -> 13만*13만 = 약 180억개의 원소

이는 메모리의 한계에 부딪히게 되며, 설령 메모리가 넉넉하더라도 대부분이 0인 배열들을 다루는 것은 비효율적. 

-> Sparse Matrix 등장

Sparse Matrix 는 값을 갖는 원소들만을 활용해서 메모리를 최적화함. 

![SparseMatrix](../static/img_8.png)

물론 Sparse Matrix가 다 좋은 것만은 아님. 

값을 불러오거나 읽을 때마다 값이 존재하는지 확인하고 그에 대한 처리도 모두 해줘야하기 때문에 데이터 처리에 대한 오버헤드가 더 커지게 됨.

따라서, 데이터가 희박하지 않은 경우에는 굳이 Sparse Matrix를 사용할 필요는 없음.

## Sparse Matrix 변환 방법

예를 들어 이런 행렬이 있다고 가정.

```python
A =
[ 0 0 3 0 ]
[ 4 0 0 0 ]
[ 0 0 0 5 ]
```

**1. COO (Coordinate Format)**

🔹 핵심 아이디어

“값이 있는 위치 (행, 열, 값)를 그대로 저장하자”

🔹 어떻게 저장할까?

위 행렬을 COO로 표현하면 이렇게 됨.

```text
row	col	value
0	2	3
1	0	4
2	3	5
```

즉, 3개의 배열로 저장.

```text
row   = [0, 1, 2]
col   = [2, 0, 3]
data  = [3, 4, 5]
```

-> (row[i], col[i]) 위치에 data[i] 값이 있다

**2. CSR (Compressed Sparse Row)**

🔹 핵심 아이디어

“행(row) 기준으로 데이터를 압축해서 저장하자”

COO보다 연산에 훨씬 유리한 구조.

🔹 CSR은 3개 배열을 사용

```text
data     : 0이 아닌 값들
indices  : 각 값의 열(col) 인덱스
indptr   : 각 행이 시작되는 위치
```

🔹 예제로 이해해보기

① data (값)

data = [3, 4, 5]

② indices (열 인덱스)

indices = [2, 0, 3]

③ indptr (행 포인터)

indptr = [0, 1, 2, 3]

```text
행 0 → data[0:1] → [3]
행 1 → data[1:2] → [4]
행 2 → data[2:3] → [5]
```

-> indptr[i]부터 indptr[i+1]까지가 i번째 행


In [1]:
import numpy as np
import pandas as pd
# sparse matrix를 사용하기 위한 scipy 라이브러리
from scipy.sparse import csr_matrix

# 간단한 테스트를 위한 임시 데이터
ratings = {'user_id':[1,2,4],
           'movie_id':[2,3,7],
           'rating':[4,3,1]}
ratings = pd.DataFrame(ratings)

# Pandas pivot을 이용해서 full matrix 변환
# 일반적인 DataFrame의 pivot 기능을 사용해서 full matrix 변환
rating_matrix = ratings.pivot(index='user_id',
                               columns='movie_id',
                               values='rating').fillna(0)
full_matrix1 = np.array(rating_matrix)
print(full_matrix1)

[[4. 0. 0.]
 [0. 3. 0.]
 [0. 0. 1.]]


In [2]:
rating_matrix

movie_id,2,3,7
user_id,,,
1,4.0,0.0,0.0
2,0.0,3.0,0.0
4,0.0,0.0,1.0


In [3]:
# Sparse matrix를 이용해서 full matrix 변환
# 원소의 값(평점) 지정
data = np.array(ratings['rating'])
# row 인덱스 지정
row_indices = np.array(ratings['user_id'])
# column 인덱스 지정
col_indices = np.array(ratings['movie_id'])
# 원래 데이터를 아까 설명했던 csr_matrix로 변환
rating_matrix = csr_matrix((data,(row_indices,col_indices)),dtype=int)
print(rating_matrix)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3 stored elements and shape (5, 8)>
  Coords	Values
  (1, 2)	4
  (2, 3)	3
  (4, 7)	1


In [4]:
full_matrix2 = rating_matrix.toarray()
print(full_matrix2)

[[0 0 0 0 0 0 0 0]
 [0 0 4 0 0 0 0 0]
 [0 0 0 3 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1]]


# 8.2. Sparse Matrix를 추천 알고리즘에 적용하기

In [ ]:
import os
import pandas as pd
base_src = '../data/'
ratings_20m_src = os.path.join(base_src,'ratings-20m.csv')
r_cols = ['user_id','movie_id','rating','timestamp']
# 20M data 읽어오기
ratings = pd.read_csv(ratings_20m_src,
                      names=r_cols,
                      sep=',',
                      encoding='latin-1')

R_temp = ratings.pivot(index='user_id',columns='movie_id',values='rating').fillna(0)


# -> ValueError: Unstacked DataFrame is too big, causing int32 overflow
# Python에서 처리할 수 없는 메모리 크기. Sparse Matrix를 사용해야함.

In [ ]:
# Sparse matrix 사용을 위한 라이브러리
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
import os
import numpy as np
import pandas as pd

base_src = '../data/'
ratings_20m_src = os.path.join(base_src,'ratings-20m.csv')
r_cols = ['user_id','movie_id','rating','timestamp']
# 20M data 읽어오기
ratings = pd.read_csv(ratings_20m_src,
                      names=r_cols,
                      sep=',',
                      encoding='latin-1')
ratings = ratings[['user_id','movie_id','rating']].astype(int)

# 데이터 지정
data = np.array(ratings['rating'])
# row 인덱스 지정
row_indices = np.array(ratings['user_id'])
# column 인덱스 지정
col_indices = np.array(ratings['movie_id'])
# csr_matrix 형식으로 데이터를 변환해서 ratings에 저장한다.
R_temp = csr_matrix((data,(row_indices,col_indices)),dtype=int)

class NEW_MF():
  def __init__(self,ratings,hyper_params):
    self.R = ratings
    self.num_users,self.num_items = np.shape(self.R)

    self.K = hyper_params['K']
    self.alpha = hyper_params['alpha']
    self.beta = hyper_params['beta']
    self.iterations = hyper_params['iterations']
    self.verbose = hyper_params['verbose']


  def rmse(self):
    xs, ys = self.R.nonzero()
    self.predictions = []
    self.errors = []
    for x,y in zip(xs,ys):
      prediction = self.get_prediction(x,y)
      self.predictions.append(prediction)
      self.errors.append(self.R[x,y] - prediction)
    self.predictions = np.array(self.predictions)
    self.errors = np.array(self.errors)
    return np.sqrt(np.mean(self.errors**2))

  def sgd(self):
    for i,j,r in self.samples:
      prediction = self.get_prediction(i,j)
      e = (r - prediction)

      self.b_u[i] += self.alpha * (e - (self.beta * self.b_u[i]))
      self.b_d[j] += self.alpha * (e - (self.beta * self.b_d[j]))

      self.P[i,:] += self.alpha * ((e * self.Q[j,:]) - (self.beta * self.P[i,:]))
      self.Q[j,:] += self.alpha * ((e * self.P[i,:]) - (self.beta * self.Q[j,:]))
 
  def get_prediction(self,i,j):
    prediction = self.b + self.b_u[i] + self.b_d[j] + self.P[i,:].dot(self.Q[j,:].T)
    return prediction

  def set_test(self,ratings_test):
    test_set = []
    for i in range(len(ratings_test)):
      x,y,z = ratings_test.iloc[i]
      test_set.append([x,y,z])
      self.R[x,y] = 0
    self.test_set = test_set
    return test_set

  def test_rmse(self):
    error = 0
    for one_set in self.test_set:
      predicted = self.get_prediction(one_set[0],one_set[1])
      error += pow(one_set[2] - predicted,2)
    return np.sqrt(error/len(self.test_set))
 
  def test(self):
    self.P = np.random.normal(scale=1./self.K,
                              size=(self.num_users,self.K))
    self.Q = np.random.normal(scale=1./self.K,
                              size=(self.num_items,self.K))
    self.b_u = np.zeros(self.num_users)
    self.b_d = np.zeros(self.num_items)
    self.b = np.mean(self.R[self.R.nonzero()])

    rows, columns = self.R.nonzero()
    self.samples = [(i,j,self.R[i,j]) for i,j in zip(rows, columns)]

    training_process = []
    for i in range(self.iterations):
      np.random.shuffle(self.samples)
      self.sgd()
      rmse1 = self.rmse()
      rmse2 = self.test_rmse()
      training_process.append((i+1,rmse1,rmse2))
      if self.verbose:
        if (i+1) % 10 == 0:
          print("Iteration : %d ; Train RMSE = %.4f ; Test RMSE = %.4f" % (i+1,rmse1,rmse2))
   
    return training_process

  def get_one_prediction(self,user_id,item_id):
    return self.get_prediction(user_id,item_id)

  def full_prediction(self):
    return self.b + self.b_u[:,np.newaxis] + self.b_d[np.newaxis,:] + self.P.dot(self.Q.T)


ratings_train,ratings_test = train_test_split(ratings,
                                              test_size=0.2,
                                              shuffle=True,
                                              random_state=2021)
     
hyper_params = {
    "K":30,
    "alpha":0.001,
    "beta":0.02,
    "iterations":100,
    "verbose":True
}

mf = NEW_MF(R_temp,hyper_params)
test_set = mf.set_test(ratings_test)
result = mf.test()